In [6]:
EMBED_MODEL = "embeddinggemma:latest"   
#EMBED_MODEL = "nomic-embed-text"

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

# Toggle downloads off if you want to stay strictly offline (then we'll use tiny built-in samples).
DOWNLOAD_FROM_WEB = True

# A small selection of long classics (public domain). Even 2–3 will give you 300+ chunks.

GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",

}


GUTENBERG = [
    (title, url) for title, url in GUTENBERG_BOOKS.items()
]

CORPUS_DIR = "corpus_jupyter"

In [13]:
import os, re, json, textwrap
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import ollama
import faiss


# Minimal helper: strip Project Gutenberg boilerplate if found.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt: str) -> str:
    start = START_MARK.search(txt)
    end = END_MARK.search(txt)
    if start and end and end.start() > start.end():
        return txt[start.end():end.start()].strip()
    # Fallback heuristics
    txt = re.sub(r"(?s)^.*?Project Gutenberg.*?eBook.*?\n", "", txt, flags=re.I)
    txt = re.sub(r"(?s)End of Project Gutenberg.*$", "", txt, flags=re.I)
    return txt.strip()

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

In [14]:
# === Check if pre-built artifacts exist; if so, load and skip heavy work ===
import json as _json

#ARTIFACTS_DIR = "C:\\Users\\arunk\\OneDrive\\Documents\\GenAI - Scaler\\LLM\\RAG\\rag_artifacts"
ARTIFACTS_DIR = "rag_artifacts"

_artifact_files = ["chunks.json", "embeddings.npy", "faiss_index.bin", "config.json"]

ARTIFACTS_LOADED = all((Path(ARTIFACTS_DIR) / f).exists() for f in _artifact_files)

if ARTIFACTS_LOADED:
    print("✅ Found pre-built artifacts! Loading from disk …")
    with open(Path(ARTIFACTS_DIR) / "chunks.json", encoding="utf-8") as _f:
        chunks = _json.load(_f)
    emb = np.load(str(Path(ARTIFACTS_DIR) / "embeddings.npy"))
    index = faiss.read_index(str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))
    print(f"   {len(chunks)} chunks | embeddings {emb.shape} | FAISS {index.ntotal} vectors")
    print("   Skipping download, chunking, embedding, and index-build cells.")
else:
    print("⏳ Artifacts not found — will build from scratch.")

⏳ Artifacts not found — will build from scratch.


In [15]:
if not ARTIFACTS_LOADED:
    Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

    docs = []  # list of dicts: {title, text, path}

    if DOWNLOAD_FROM_WEB:
        for title, url in GUTENBERG:
            out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"
            if not out_path.exists():
                print(f"Downloading: {title}")
                try:
                    r = requests.get(url, timeout=60)
                    r.raise_for_status()
                    clean = strip_gutenberg_boilerplate(r.text)
                    out_path.write_text(clean, encoding="utf-8")
                except Exception as e:
                    print(f"  Failed ({e}); skipping.")
            else:
                print(f"Exists: {out_path.name}")
            if out_path.exists():
                docs.append({
                    "title": title,
                    "text": out_path.read_text(encoding="utf-8", errors="ignore"),
                    "path": str(out_path)
                })

    print(f"Loaded {len(docs)} docs")
else:
    print("⏩ Skipped (artifacts already loaded)")

Exists: Moby-Dick.txt
Exists: Pride_and_Prejudice.txt
Exists: Frankenstein.txt
Exists: Alice_in_Wonderland.txt
Exists: Dracula.txt
Exists: A_Tale_of_Two_Cities.txt
Exists: The_Great_Gatsby.txt
Exists: Adventures_of_Sherlock_Holmes.txt
Exists: War_and_Peace.txt
Exists: Jane_Eyre.txt
Exists: The_Picture_of_Dorian_Gray.txt
Exists: Crime_and_Punishment.txt
Exists: Wuthering_Heights.txt
Loaded 13 docs
